# Speech Models

**Module:** 16 — Speech AI

OpenAI, Google, Meta, Microsoft, and open-source speech stacks — selection guide.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare major ASR/TTS/audio LLM offerings
- Choose batch vs streaming vs on-device models
- Apply a selection rubric for cost, WER, latency, residency
- Route traffic across models by locale and channel


## Landscape

### Definition
Speech models include cloud ASR/TTS APIs, multimodal audio LLMs, and open weights for self-host/on-device.

### Why it matters
Telephony, on-device privacy, and batch captioning have different winners.

### How it works
Evaluate on your channel (8 kHz phone vs 16 kHz app), locales, and streaming needs.

### Intuition
Pick the ear/mouth that fits the room you're in.

### Pitfalls
- One vendor for all locales blindly
- Ignoring telephony encoding

### When to use
Architecture and procurement.


### Provider snapshot (verify current docs)

| Vendor | Strengths | Typical use |
|--------|-----------|-------------|
| OpenAI | Whisper / TTS / realtime audio | Apps, agents, prototypes |
| Google | Speech-to-Text, WaveNet/Journey TTS | GCP + telephony |
| Meta | Seamless / MMS research & open | Multilingual / research |
| Microsoft | Azure Speech (STT/TTS/avatar) | Enterprise + call centers |
| Open | Whisper, Coqui, VITS, Piper… | Large volume / air-gap |

```mermaid
flowchart LR
  J[Job] --> C{Constraints}
  C -->|airgap| O[Open weights]
  C -->|phone| T[Telephony-grade STT/TTS]
  C -->|agent duplex| R[Realtime API / S2S]
  C -->|batch captions| B[Batch ASR]
```


In [ ]:
# Demo 1: selection rubric
from dataclasses import dataclass

@dataclass
class SpeechModel:
    name: str; wer: float; latency: float; cost: float; streaming: bool; on_device: bool
    def score(self, w):
        s = -w["wer"]*self.wer - w["latency"]*self.latency - w["cost"]*self.cost
        if w.get("need_stream") and self.streaming: s += 0.3
        if w.get("need_on_device") and self.on_device: s += 0.4
        return s

models = [
    SpeechModel("cloud-stream", 0.08, 0.3, 0.6, True, False),
    SpeechModel("batch-whisper-large", 0.06, 0.9, 0.4, False, False),
    SpeechModel("on-device-small", 0.14, 0.2, 0.1, True, True),
]
w = {"wer":0.4,"latency":0.25,"cost":0.2,"need_stream":True,"need_on_device":False}
print(sorted(((m.score(w), m.name) for m in models), reverse=True))


In [ ]:
# Demo 2: locale/channel router
def route_asr(job: dict) -> str:
    if job.get("on_device"): return "whisper-tiny-local"
    if job.get("channel") == "phone": return "telephony-enhanced-asr"
    if job.get("lang") not in {"en", "en-US"}: return "multilingual-asr"
    if job.get("mode") == "batch": return "whisper-batch"
    return "cloud-streaming-asr"
for j in [
    {"channel":"phone","lang":"en"},
    {"mode":"batch","lang":"en"},
    {"lang":"hi"},
    {"on_device":True},
]:
    print(j, "->", route_asr(j))


In [ ]:
# Demo 3: cost estimate hours of audio
def asr_cost(hours, price_per_min=0.006):
    return hours * 60 * price_per_min
def tts_cost(chars, price_per_m_chars=15.0):
    return chars / 1e6 * price_per_m_chars
print("1000h ASR $", round(asr_cost(1000), 2))
print("5e6 chars TTS $", round(tts_cost(5_000_000), 2))


## Selection Guide

### Definition
Selection balances WER/MOS, streaming latency, languages, residency, offline needs, and price.

### Why it matters
Wrong tier shows up as either bill shock or failed calls.

### How it works
Freeze an eval set per channel; measure partial lag + WER; decide cloud vs edge; plan fallbacks.

### Intuition
A race car and a tractor are both vehicles — pick for terrain.

### Pitfalls
- Chasing leaderboard WER on audiobooks for call centers
- No fallback when primary 429s

### When to use
Every production speech stack.


In [ ]:
# Demo 4: bakeoff table printer
rows = [
    {"model": "A", "wer": 0.09, "p90_partial_ms": 350, "$/hr": 0.36},
    {"model": "B", "wer": 0.07, "p90_partial_ms": 700, "$/hr": 0.48},
    {"model": "C", "wer": 0.12, "p90_partial_ms": 200, "$/hr": 0.10},
]
# prefer WER then latency
ranked = sorted(rows, key=lambda r: (r["wer"], r["p90_partial_ms"]))
for r in ranked:
    print(r)


In [ ]:
# Demo 5: Microsoft/Google-style config sketch (illustrative)
import json
azureish = {
    "subscription_key": "YOUR_AZURE_SPEECH_KEY",
    "region": "eastus",
    "language": "en-US",
    "enable_dictation": True,
    "profanity_option": "Masked",
}
googleish = {
    "config": {"languageCode": "en-US", "enableWordTimeOffsets": True, "model": "telephony"},
    "audio": {"uri": "gs://bucket/call.wav"},
}
print(json.dumps(azureish, indent=2))
print(json.dumps(googleish, indent=2))


In [ ]:
# Demo 6: fallback chain
class FallbackASR:
    def __init__(self, providers):
        self.providers = providers  # list of callables
    def transcribe(self, audio_ref):
        errors = []
        for p in self.providers:
            try:
                return p(audio_ref)
            except Exception as e:
                errors.append(str(e))
        raise RuntimeError("all failed: " + "; ".join(errors))

def primary(_): raise TimeoutError("429/timeout")
def secondary(x): return {"text": f"transcript of {x}", "provider": "secondary"}
print(FallbackASR([primary, secondary]).transcribe("call.wav"))


### OpenAI vs others (decision hints)

| If you need… | Consider |
|--------------|----------|
| Fast app prototype | OpenAI Whisper/TTS/Realtime |
| Carrier telephony | Azure/Google telephony models |
| On-prem only | Open Whisper/Piper etc. |
| Many languages | Multilingual clouds / Meta-lineage open |
| Speech-to-speech | Realtime S2S APIs |


### Checklist — Model selection

- [ ] Eval audio from production channel
- [ ] Streaming lag measured
- [ ] Locale coverage confirmed
- [ ] Residency/compliance OK
- [ ] Fallback provider configured


### Try it yourself — Speech models

1. Reweight the rubric for on-device dictation.
2. Add a Brazilian Portuguese routing rule.
3. Estimate monthly $ for 50k call-minutes.

**Stretch:** Run the same 10 clips across two providers; compare WER.


### Try it yourself — Operations

1. Design a circuit breaker for ASR timeouts.
2. Document data retention differences by vendor.


## Knowledge Check

**Q1.** When is batch Whisper a poor fit?

<details><summary>Answer</summary>

Live duplex voice agents needing partials and low time-to-final.

</details>

**Q2.** Why test telephony audio separately?

<details><summary>Answer</summary>

8 kHz codecs and channel noise shift WER vs clean app audio.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `Whisper` | OpenAI ASR model family (also open weights variants) |
| `telephony model` | ASR tuned for phone channels |
| `on-device` | Runs locally on phone/edge |
| `fallback` | Secondary provider on failure |
| `MOS` | TTS listening quality score |


## Key Takeaways

- Match model tier to channel and streaming needs
- Rubrics beat brand loyalty
- Always keep a fallback path
- Price hours/minutes explicitly


## Production Incident Patterns — speech models

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "speech models",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — speech models

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("speech models", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — speech models ops

1. Draft an on-call runbook bullet list for speech models when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
